# **Sentiment Analysis of Real-time Flipkart Product Reviews**

The objective of this project is to classify customer reviews as positive or negative and understand the pain points of customers who write negative reviews. By analyzing the sentiment of reviews, we aim to gain insights into product features that contribute to customer satisfaction or dissatisfaction.


## **1. EXPLORING THE DATA**

In [ ]:
# Importing Libraries

import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt

In [ ]:
# Load The Dataset

badminton_df = pd.read_csv("D:\Users\yashs\Downloads\Flipcart Product Review App\data\reviews_badminton.csv")
tawa_df = pd.read_csv("D:\Users\yashs\Downloads\Flipcart Product Review App\data\reviews_tawa.csv")
tea_df = pd.read_csv("D:\Users\yashs\Downloads\Flipcart Product Review App\data\reviews_tea.csv")

In [ ]:
# Data Exploration
# Explore badminton_df First

badminton_df

In [ ]:
badminton_df.info()

In [ ]:
badminton_df.describe()

In [ ]:
badminton_df.isnull().sum()

In [ ]:
# Explore tawa_df Second

tawa_df.info()

In [ ]:
tawa_df

In [ ]:
tawa_df.describe()

In [ ]:
tawa_df.isnull().sum()

In [ ]:
# Explore tea_df Third

tea_df

In [ ]:
tea_df.info()

In [ ]:
tea_df.describe()

In [ ]:
tea_df.isnull().sum()

In [ ]:
print(badminton_df.columns)
print(tawa_df.columns)
print(tea_df.columns)

## **2. DATA PREPROCESSING**

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

In [ ]:
combined_df = pd.concat([badminton_df, tawa_df, tea_df])

In [ ]:
# Text Cleaning

def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9]", " ", str(text))
    text = text.lower()
    return text

In [ ]:
# Text Normalization

def normalize_text(text):
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    normalized_words = [lemmatizer.lemmatize(word) for word in words]
    normalized_text = " ".join(normalized_words)
    return normalized_text

In [ ]:
# Combine Clean & Normalization

def preprocess_text(text):
    cleaned_text = clean_text(text)
    normalized_text = normalize_text(cleaned_text)
    return normalized_text

In [ ]:
combined_df['processed_text'] = combined_df['review_text'].apply(preprocess_text)

In [ ]:
# Checking all Preprocess Data

print("Badminton Data:")
print(badminton_df.head())

In [ ]:
print("\nTawa Data:")
print(tawa_df.head())

In [ ]:
print("\nTea Data:")
print(tea_df.head())

## **3. SENTIMENT ANALYSIS**

In [ ]:
combined_df = pd.concat([badminton_df, tawa_df, tea_df])

In [ ]:
sid = SentimentIntensityAnalyzer()

In [ ]:
combined_df['review_text'] = combined_df['review_text'].astype(str)

In [ ]:
combined_df = combined_df.dropna(subset=['review_text'])

In [ ]:
def get_sentiment_score(text):
    return sid.polarity_scores(text)['compound']

In [ ]:
combined_df['sentiment_score'] = combined_df['review_text'].apply(get_sentiment_score)

In [ ]:
combined_df['sentiment'] = combined_df['sentiment_score'].apply(lambda score: 'Positive' if score > 0 else 'Negative')

In [ ]:
print(combined_df[['review_text', 'sentiment_score', 'sentiment']].head())

In [ ]:
sentiment_counts = combined_df['sentiment'].value_counts()
plt.figure(figsize=(8, 6))
sentiment_counts.plot(kind='bar', color=['green', 'red'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.show()

In [ ]:
sentiment_counts = combined_df['sentiment'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', colors=['lightgreen', 'lightcoral'], startangle=140)
plt.title('Sentiment Distribution')
plt.axis('equal')
plt.show()

## **4. MODEL TRAINING**

In [ ]:
# Import Libraries For Model Training

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pickle

In [ ]:
X = combined_df['review_text']
y = combined_df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

In [ ]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

In [ ]:
X_test_tfidf = tfidf_vectorizer.transform(X_test)
y_pred = model.predict(X_test_tfidf)
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

In [ ]:
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1-Score: {f1}")
print(f"F1-Score: {f1*100}")

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump((model, tfidf_vectorizer), f)

## **5. MODEL DEPLOYMENT**

In [ ]:
from flask import Flask, render_template, request
import pickle

app = Flask(__name__)

In [ ]:
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

In [ ]:
def preprocess_text(text):
    return text
def predict_sentiment(text):
    text_processed = preprocess_text(text)
    prediction = model.predict([text_processed])[0]
    sentiment = 'Positive' if prediction == 1 else 'Negative'
    return sentiment

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if request.method == 'POST':
        text = request.form['text']
        sentiment = predict_sentiment(text)
        return render_template('result.html', text=text, sentiment=sentiment)

if __name__ == '__main__':
    app.run(debug=True)